## Imports

In [ ]:
import json
from pathlib import Path
import re
import warnings
from tqdm.auto import tqdm
from datasets import load_dataset, get_dataset_config_names, get_dataset_split_names
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas.api.types import is_datetime64_any_dtype
from dateutil.parser import parse
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
import nltk

nltk.download("stopwords", quiet=True)

# Explicitly required to unlock HalvingRandomSearchCV in scikit-learn
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_classif, f_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    HalvingRandomSearchCV,
    KFold,
    StratifiedKFold,
    TimeSeriesSplit,
    cross_validate,
    train_test_split,
)
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    PolynomialFeatures,
    StandardScaler,
    FunctionTransformer,
)
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from xgboost import XGBClassifier, XGBRegressor
from scipy import stats
from statsmodels.stats.stattools import durbin_watson
from sklearn.metrics import matthews_corrcoef

warnings.filterwarnings("ignore")

## Directory

In [ ]:
# Insert dataset name here: (if configuration name doesn't exist leave blank)
# ------------------------------------------------------------------------------------------------------------
DATASET_NAME = ""
DATASET_CONFIGURATION = ""
SPLIT = "train"
# ------------------------------------------------------------------------------------------------------------
RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd()
# Supports launching Jupyter from either the repository root or notebooks/.
if not (PROJECT_ROOT / "src").is_dir() and (PROJECT_ROOT.parent / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
RUN_SLUG = DATASET_NAME.lower().replace("/", "-").strip("-._") or "dataset"

RUN_ID = f"reg_class_platform_{RUN_SLUG}"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "runs" / RUN_ID
CONFIG_DIR = ARTIFACT_DIR / "config"
CHECKPOINT_DIR = ARTIFACT_DIR / "checkpoints"
MANIFEST_DIR = ARTIFACT_DIR / "manifests"
METRICS_DIR = ARTIFACT_DIR / "metrics"
FIGURE_DIR = ARTIFACT_DIR / "figures"
REPORT_DIR = ARTIFACT_DIR / "reports"
for directory in (
    CONFIG_DIR,
    CHECKPOINT_DIR,
    MANIFEST_DIR,
    METRICS_DIR,
    FIGURE_DIR,
    REPORT_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

## Dataset

In [ ]:
if DATASET_NAME:
    configs = get_dataset_config_names(DATASET_NAME)[0]
    splits = get_dataset_split_names(DATASET_NAME, config_name=configs)[0]
    try:
        dataset = load_dataset(
            DATASET_NAME,
            DATASET_CONFIGURATION or configs or None,
            split=SPLIT or splits or None,
        )
    except Exception:
        dataset = None

    for delimiter in [",", "\t", ";", "|", " "]:
        try:
            dataset = load_dataset(
                "csv",
                data_files=DATASET_NAME,
                split=SPLIT or splits or None,
                delimiter=delimiter,
            )
            break
        except Exception:
            continue
    if dataset is None:
        raise ValueError("The dataset could not be loaded or does not exist.\n")
else:
    raise ValueError("Insert dataset name in the Directory section")

ticker_names = {
    "ticker",
    "symbol",
    "pair",
    "stock_id",
    "tickers",
    "symbols",
    "pairs",
    "stock_ids",
    "stockname",
    "stock_name",
    "stocknames",
    "stock_names",
}
ticker_columns = [
    column for column in dataset.column_names if column.lower() in ticker_names
]

if ticker_columns:
    ticker_column = ticker_columns[0]

    ticker_options = sorted(dataset.unique(ticker_column))

    if ticker_options:
        selected_ticker = ticker_options[0]
        print("Selected ticker:", selected_ticker)

        dataset = dataset.filter(lambda row: row[ticker_column] == selected_ticker)

dataset_df = dataset.to_pandas()

print("Dataset Preview:")
print(dataset_df.head(10))
print(f"\nDataset Dimensions:{dataset_df.shape[0]} rows, {dataset_df.shape[1]} columns")
print("\nData Types:")
print(", ".join([f"{col} ({dtype})" for col, dtype in dataset_df.dtypes.items()]))

saved_paths_dataset = MANIFEST_DIR / "dataset_original.csv"
dataset_df.to_csv(saved_paths_dataset, index=True)
print("\nOriginal dataset saved successfully. Generated artifacts:")
print(f"  • {saved_paths_dataset.name}")

## Target Column Selection

In [ ]:
def select_target_column(dataset):

    print("\nAvailable columns:\n")

    available_columns = dataset.columns.tolist()

    for position, column_name in enumerate(
        available_columns,
        start=1,
    ):
        print(f"{position}: {column_name}")

    # Continue asking until the user enters a valid selection.
    while True:
        try:
            selection = int(input("\nSelect the target column number: "))

            if not 1 <= selection <= len(available_columns):
                raise ValueError

            target_column = available_columns[selection - 1]

            y = dataset[target_column].copy()
            break

        except ValueError:
            print("Invalid selection. Enter one of the numbers displayed above.")

    print(f"\nSelected target column: {target_column}")

    return target_column


target_column = select_target_column(dataset_df)

## Project Type Detection

### Classification and Regresion Analysis

In [ ]:
def detect_project_type(dataset, target_col, unique_ratio_thresh=0.05):

    dataset_df = dataset.copy()
    # 1. Clean missing values
    target = dataset_df[target_col].dropna()
    N = len(target)
    is_classification = False
    is_regression = False

    if dataset_df.shape[0] < 5 or dataset_df.shape[1] < 2:
        raise ValueError(
            "The dataset is not suitable for regression or classification. The minimum dimensional requirements are: 5 rows and 2 columns."
        )

    if N == 0:
        raise ValueError("Undertemined type of project. The target column is empty")

    # 2. Get unique counts and ratios
    unique_vals = target.unique()
    U = len(unique_vals)
    R = U / N

    # 3. Check text/categorical types
    if (
        pd.api.types.is_object_dtype(target)
        or pd.api.types.is_string_dtype(target)
        or pd.api.types.is_bool_dtype(target)
        or isinstance(target.dtype, pd.CategoricalDtype)
    ):

        # Safeguard against free-form text
        if pd.api.types.is_object_dtype(target) or pd.api.types.is_string_dtype(target):
            text_values = target.astype(str)

            avg_words = text_values.str.split().str.len().mean()
            avg_chars = text_values.str.len().mean()

            if avg_words > 5 or avg_chars > 50:
                raise ValueError(
                    "The selected target column appears to contain free-form text. "
                    "It is not suitable as a standard classification or regression target. "
                    "Please select a categorical or numeric target column."
                )

        if U == N and N > 10:
            raise ValueError(
                "Undertemined type of project. The target column contains unique text"
            )

        is_classification = True
        is_regression = False
        project_type = "Classification"
        return is_classification, is_regression, project_type

    # 4. Check numeric types
    if np.issubdtype(target.dtype, np.number):
        is_int_like = np.all(np.equal(np.mod(target, 1), 0))
        if U == 2:
            is_classification = True
            is_regression = False
            project_type = "Classification"
            return is_classification, is_regression, project_type

        # If the numbers are heavily repeated (low cardinality) or unique count is very small
        elif (R < unique_ratio_thresh or U <= 15) and is_int_like:
            is_classification = True
            is_regression = False
            project_type = "Classification"
            return is_classification, is_regression, project_type
        else:
            is_classification = False
            is_regression = True
            project_type = "Regression"
            return is_classification, is_regression, project_type
    else:
        raise ValueError(
            "Undertemined type of project. The target column contains unrecognized data type"
        )


is_classification, is_regression, project_type = detect_project_type(
    dataset_df, target_column, unique_ratio_thresh=0.05
)


print(f"Target column analysis indentifies the project as a {project_type}.")
print("Regression:", is_regression)
print("Classification:", is_classification)


# Checks if it is a Natural Language Processing project
def nlp_check(dataset, high_cardinality_threshold=0.90, free_text_average_length=90):

    dataset_df = dataset.copy()
    nlp_column = None
    nlp_flag = False

    if is_classification:
        for column_name in dataset_df.columns:
            column = dataset_df[column_name]

            nonmissing_count = column.notna().sum()

            unique_count = column.nunique(dropna=True)

            uniqueness_ratio = (
                unique_count / nonmissing_count if nonmissing_count > 0 else 0.0
            )
            if (
                pd.api.types.is_object_dtype(column)
                or isinstance(
                    column.dtype,
                    pd.StringDtype,
                )
                or isinstance(
                    column.dtype,
                    pd.CategoricalDtype,
                )
            ):
                text_values = column.dropna().astype(str)

                average_text_length = (
                    text_values.str.len().mean() if not text_values.empty else 0
                )

                if (
                    average_text_length >= free_text_average_length
                    and uniqueness_ratio >= high_cardinality_threshold
                ):
                    if dataset_df.shape[1] <= 3:

                        nlp_flag = True
                        nlp_column = column.name
                        print(
                            "\nAdditional dataset analysis identified a Natural Language Processing requirement."
                        )

    return nlp_flag, nlp_column


nlp_flag, nlp_column = nlp_check(
    dataset_df, high_cardinality_threshold=0.90, free_text_average_length=90
)

### Time-Series Analysis

In [ ]:
# Checks if dataset contains date columns and sorts the dataset by date after converting the column to datetime.
def time_series_check(
    dataset,
    series,
    *,
    maximum_lag=30,
    minimum_correlation=0.20,
):

    dataset_df = dataset.copy()
    is_autocorrelated = False
    is_time_series_candidate = False
    is_timeseries = False
    is_shuffle = True
    is_crosssectional = True
    selected_date_column = None
    date_time_pattern = re.compile(
        r"date|time|timestamp|datetime|created_at",
        flags=re.IGNORECASE,
    )
    date_columns = []

    # Find possible date/time columns.
    for column_name in dataset_df.columns:
        column = dataset_df[column_name]

        if is_datetime64_any_dtype(column):

            date_columns.append(column_name)
            continue

        elif date_time_pattern.search(str(column_name)):
            sample_values = column.dropna().astype(str).unique()[:10]
            date_finder = False
            for val in sample_values:
                if val.replace(".", "", 1).isdigit():
                    continue
                try:
                    parse(val)
                    date_finder = True  # Set flag to True if at least one sample parses successfully
                    break  # Stop checking rows for this column once confirmed
                except Exception:
                    pass  # Ignore parsing errors and try the next sample row
            if date_finder:
                date_columns.append(column_name)

    has_date = bool(date_columns)

    if has_date:
        print("Available date/time columns:\n")

        for position, column_name in enumerate(
            date_columns,
            start=1,
        ):
            print(f"{position}: {column_name}")

        # Ask the user to select a date column.
        while True:
            try:
                selection = int(input("\nSelect the date column number: "))

                if not 1 <= selection <= len(date_columns):
                    raise ValueError

                selected_date_column = date_columns[selection - 1]

                break

            except ValueError:
                print("Invalid selection. Enter one of the numbers displayed above.")

        dataset_df[selected_date_column] = pd.to_datetime(
            dataset_df[selected_date_column], format="mixed", errors="coerce"
        )
        sorted_index = dataset_df.sort_values(
            by=selected_date_column,
            kind="mergesort",
            na_position="last",
        ).index

        # Sort the dataset and target using the same order.
        dataset_df = dataset_df.loc[sorted_index].reset_index(drop=True)

        series = series.loc[sorted_index].reset_index(drop=True)

        # Remove invalid target values.
        if pd.api.types.is_numeric_dtype(series):
            valid_series = series.notna() & np.isfinite(series)

            series = series.loc[valid_series].reset_index(drop=True)

            # Test autocorrelation when enough values are available.
            if len(series) > 1:
                effective_maximum_lag = min(
                    maximum_lag,
                    len(series) - 1,
                )

                for lag in range(1, effective_maximum_lag + 1):

                    correlation = series.autocorr(lag=lag)

                    if (
                        pd.notna(correlation)
                        and abs(correlation) >= minimum_correlation
                    ):
                        is_autocorrelated = True
                        break

            for frequency, date_col in [
                ("D", "_date"),
                ("h", "_hour"),
                ("min", "_minute"),
            ]:
                dataset_df[date_col] = dataset_df[selected_date_column].dt.floor(
                    frequency
                )

                grouped_series = (
                    dataset_df.groupby(date_col)[target_column].mean().sort_index()
                )

                if len(grouped_series) >= 3:
                    selected_series = grouped_series
                    break
            else:
                raise ValueError(
                    "Not enough observations after grouping by minute, hour or day."
                )

        elif pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(
            series
        ):

            # Remove invalid target values.
            valid_series = series.notna()
            series = series.loc[valid_series].reset_index(drop=True)

            if len(series) > 1:
                effective_maximum_lag = min(maximum_lag, len(series) - 1)

                for lag in range(1, effective_maximum_lag + 1):
                    earlier_labels = series.iloc[:-lag].reset_index(drop=True)
                    later_labels = series.iloc[lag:].reset_index(drop=True)

                    agreement = earlier_labels.eq(later_labels).mean()

                    # Compare agreement with the rate expected from class frequencies.
                    expected_agreement = (
                        series.value_counts(normalize=True).pow(2).sum()
                    )

                    if (
                        pd.notna(agreement)
                        and agreement - expected_agreement >= minimum_correlation
                    ):
                        is_autocorrelated = True
                        break

            # Group observations by time bucket, using the most common class per bucket.
            for frequency, date_col in [
                ("D", "_date"),
                ("h", "_hour"),
                ("min", "_minute"),
            ]:
                dataset_df[date_col] = dataset_df[selected_date_column].dt.floor(
                    frequency
                )

                grouped_series = (
                    dataset_df.groupby(date_col)[target_column]
                    .agg(
                        lambda values: (
                            values.mode().iloc[0] if not values.mode().empty else pd.NA
                        )
                    )
                    .sort_index()
                )

                if len(grouped_series) >= 3:
                    selected_series = grouped_series
                    break
            else:
                raise ValueError(
                    "Not enough observations after grouping by minute, hour or day."
                )

        is_time_series_candidate = (
            selected_series.index.is_monotonic_increasing
            and selected_series.index.is_unique
            and selected_series.notna().all()
        )
        if is_autocorrelated and is_time_series_candidate:
            is_timeseries = True
            is_shuffle = False
            is_crosssectional = False

        print("Detected date column:", has_date)
        print("Selected date column:", selected_date_column)
        print("Time series:", is_timeseries)
        print("Shuffle:", is_shuffle)
        print("Cross-sectional:", is_crosssectional)

    else:

        print("No date/time columns were found.")

    return (
        is_timeseries,
        is_shuffle,
        is_crosssectional,
        selected_date_column,
        date_columns,
        has_date,
    )


(
    is_timeseries,
    is_shuffle,
    is_crosssectional,
    selected_date_column,
    date_columns,
    has_date,
) = time_series_check(
    dataset_df,
    dataset_df[target_column],
    maximum_lag=30,
    minimum_correlation=0.20,
)

## Configurations

In [ ]:
REGRESSION_ANALYSIS = is_regression
CLASSIFICATION_ANALYSIS = is_classification
PROJECT_TYPE = project_type
NATURAL_LANGUAGE_PROCESSING = nlp_flag
TIME_SERIES = is_timeseries
CROSS_SECTIONAL = is_crosssectional
SHUFFLE = is_shuffle
DATASET_SHAPE = dataset_df.shape
DATASET_ROW_SIZE = dataset_df.shape[0]
DATASET_COLUMN_SIZE = dataset_df.shape[1]
TEST_SIZE = 0.2

print("Project Configuration:")
print("-----------------------------")
print("Regression Analysis:", REGRESSION_ANALYSIS)
print("Classification Analysis:", CLASSIFICATION_ANALYSIS)
print("Project Type:", PROJECT_TYPE)
print("Natural Language Processing:", NATURAL_LANGUAGE_PROCESSING)
print("Time-Series:", TIME_SERIES)
print("Cross-Sectional:", CROSS_SECTIONAL)
print("Shuffle:", SHUFFLE)
print("Dataset Shape:", DATASET_SHAPE)

run_config = {
    "random_state": RANDOM_STATE,
    "dataset": DATASET_NAME,
    "dataset_configuration": DATASET_CONFIGURATION,
    "split": SPLIT,
    "regression_analysis": REGRESSION_ANALYSIS,
    "classification_analysis": CLASSIFICATION_ANALYSIS,
    "project_type": PROJECT_TYPE,
    "natural_language_processing": NATURAL_LANGUAGE_PROCESSING,
    "time_series": TIME_SERIES,
    "cross_sectional": CROSS_SECTIONAL,
    "shuffle": SHUFFLE,
    "dataset_shape": DATASET_SHAPE,
    "test_size": TEST_SIZE,
}

saved_paths_config = CONFIG_DIR / "run_config.json"
saved_paths_config.write_text(json.dumps(run_config, indent=2), encoding="utf-8")
print("\nProject configuration checkpoint saved successfully. Generated artifacts:")
print(f"  • {saved_paths_config.name}")

## Dimensionality Reduction

In [ ]:
def remove_non_crucial_columns(
    dataset,
    target_column,
    *,
    protected_columns=None,
    drop_columns=None,
    high_cardinality_threshold=0.90,
    free_text_average_length=90,
):

    if not isinstance(dataset_df, pd.DataFrame):
        raise TypeError("dataset_df must be a pandas DataFrame.")

    dataset_reduction = dataset.copy()

    protected_columns = (
        {protected_columns}
        if isinstance(protected_columns, str)
        else set(protected_columns or [])
    )

    # Never remove the target itself.
    protected_columns.add(target_column)

    # Drop requested feature columns.
    if isinstance(drop_columns, str):
        drop_columns = [drop_columns]
    else:
        drop_columns = list(drop_columns or [])

    protected_requested = set(drop_columns) & set(protected_columns)

    removal = False
    if drop_columns:
        unknown = set(drop_columns) - set(dataset_reduction.columns)
        if unknown:
            raise ValueError(f"Columns not found in the dataset: {sorted(unknown)}")

        elif protected_requested:
            print(
                f"Cannot drop protected column(s): {sorted(protected_requested)}. The request was voided.\n"
            )

        elif DATASET_COLUMN_SIZE == 2:
            print(
                f"Cannot drop requested column(s) because the dataset doesn't fulfill dimensional requirements (column size: {DATASET_COLUMN_SIZE}). The request was voided.\n"
            )
        else:
            removal = True

    # ---------------------------------------------------------
    # Column-name patterns
    # ---------------------------------------------------------
    sensitive_pattern = re.compile(
        r"""
        password
        |passwd
        |passcode
        |secret
        |token
        |api[_\s-]*key
        |email
        |e[_\s-]*mail
        |phone
        |telephone
        |mobile
        |fax
        |ssn
        |social[_\s-]*security
        |first[_\s-]*name
        |last[_\s-]*name
        |full[_\s-]*name
        |customer[_\s-]*name
        |customer[_\s-]*fname
        |customer[_\s-]*lname
        |fname
        |lname
        |username
        |street
        |address
        """,
        flags=re.IGNORECASE | re.VERBOSE,
    )

    media_pattern = re.compile(
        r"""
        image
        |photo
        |picture
        |avatar
        |url
        |uri
        |filepath
        |file[_\s-]*path
        """,
        flags=re.IGNORECASE | re.VERBOSE,
    )

    identifier_pattern = re.compile(
        r"""
        (^|[_\s-])
        (
            id
            |uuid
            |guid
            |identifier
            |index
            |row[_\s-]*number
            |record[_\s-]*number
            |tracking[_\s-]*number
            |invoice[_\s-]*number
            |transaction[_\s-]*number
        )
        ($|[_\s-])
        """,
        flags=re.IGNORECASE | re.VERBOSE,
    )

    personal_identifier_pattern = re.compile(
        r"""
        customer[_\s-]*id
        |user[_\s-]*id
        |employee[_\s-]*id
        |patient[_\s-]*id
        |account[_\s-]*id
        |person[_\s-]*id
        """,
        flags=re.IGNORECASE | re.VERBOSE,
    )

    removal_records = []

    # ---------------------------------------------------------
    # Inspect every column
    # ---------------------------------------------------------

    removal_count = 0

    for column_name in dataset_reduction.columns:
        if DATASET_COLUMN_SIZE - removal_count == 2:
            if removal:
                print(
                    f"Cannot drop requested column: {column_name}. The dataset doesn't fulfill dimensional requirements (current column size: {DATASET_COLUMN_SIZE - removal_count}). The request was voided.\n"
                )
            break
        if column_name in protected_columns:
            continue

        column = dataset_reduction[column_name]
        normalized_name = str(column_name).strip()

        nonmissing_count = column.notna().sum()

        unique_count = column.nunique(dropna=True)

        uniqueness_ratio = (
            unique_count / nonmissing_count if nonmissing_count > 0 else 0.0
        )

        removal_reason = None

        # Remove user requested columns.
        if column_name in drop_columns and removal:
            removal_reason = "user manual request"

        # Remove completely empty columns.
        elif nonmissing_count == 0:
            removal_reason = "all values are missing"

        # Remove constant columns.
        elif unique_count <= 1:
            removal_reason = "constant column"

        # Remove confidential or personal information.
        elif sensitive_pattern.search(normalized_name):
            removal_reason = "personal or confidential information"

        # Remove personal identifiers.
        elif personal_identifier_pattern.search(normalized_name):
            removal_reason = "personal identifier"

        # Remove image, URL and file-reference columns.
        elif media_pattern.search(normalized_name):
            removal_reason = "image, URL or file reference"

        # Remove identifier columns only when mostly unique.
        elif (
            identifier_pattern.search(normalized_name)
            and uniqueness_ratio >= high_cardinality_threshold
        ):
            removal_reason = "high-cardinality identifier"

        # Remove long and mostly unique text.
        elif (
            pd.api.types.is_object_dtype(column)
            or isinstance(
                column.dtype,
                pd.StringDtype,
            )
            or isinstance(
                column.dtype,
                pd.CategoricalDtype,
            )
        ):
            text_values = column.dropna().astype(str)

            average_text_length = (
                text_values.str.len().mean() if not text_values.empty else 0
            )

            if (
                average_text_length >= free_text_average_length
                and uniqueness_ratio >= high_cardinality_threshold
            ):
                if DATASET_COLUMN_SIZE > 3:
                    removal_reason = "long, high-cardinality text"

        if removal_reason is not None:
            removal_count += 1
            removal_records.append(
                {
                    "column": column_name,
                    "reason": removal_reason,
                    "dtype": str(column.dtype),
                    "unique_values": unique_count,
                    "uniqueness_ratio": (uniqueness_ratio),
                    "missing_percentage": (100 * column.isna().mean()),
                }
            )

    # ---------------------------------------------------------
    # Remove identified columns
    # ---------------------------------------------------------

    if not removal_records:
        removed_columns = []
        removal_report = pd.DataFrame(
            columns=["The removal operation is empty. All columns were preserved"]
        )

    else:
        removal_report = pd.DataFrame(removal_records)
        removed_columns = removal_report["column"].tolist()

        removal_report = removal_report.sort_values(
            by=["reason", "column"]
        ).reset_index(drop=True)

    dataset_reduction = dataset_reduction.drop(columns=removed_columns)

    print(f"Original columns:  {dataset_df.shape[1]}")

    print(f"Removed columns:   {len(removed_columns)}")

    print(f"Remaining columns: " f"{dataset_reduction.shape[1]}")

    return (removal_report, removed_columns, dataset_reduction, protected_columns)


removal_report, removed_columns, dataset_reduction, protected_columns = (
    remove_non_crucial_columns(
        dataset_df,
        target_column,
        protected_columns=([selected_date_column] if is_timeseries else None),
        drop_columns=None,
        high_cardinality_threshold=0.90,
        free_text_average_length=90,
    )
)

saved_paths_dimred = []
csv_dimred_path = MANIFEST_DIR / "dimensionality_reduction_report.csv"
removal_report.to_csv(csv_dimred_path, index=False)
saved_paths_dimred.append(csv_dimred_path)
csv_pd_path = MANIFEST_DIR / "dataset_reduction.csv"
dataset_reduction.to_csv(csv_pd_path, index=False)
saved_paths_dimred.append(csv_pd_path)
print(f"\nData formatting checkpoint successfully saved. Generated artifacts:")
for path in saved_paths_dimred:
    print(f"  • {Path(path).name}")

## Time-Series Processor

In [ ]:
def time_series_processor(dataset):

    dataset_df = dataset.copy()
    if is_timeseries:
        dataset_df[selected_date_column] = pd.to_datetime(
            dataset_df[selected_date_column], errors="coerce"
        )
        sorted_index = dataset_df.sort_values(
            by=selected_date_column,
            kind="mergesort",
            na_position="last",
        ).index
        dataset_df = dataset_df.loc[sorted_index].reset_index(drop=True)

        print(f"\nDataset sorted by: {selected_date_column}\n")
        print(f"Data types:\n{dataset_df.dtypes}")

    else:
        print("The dataset doesn't require time-series processing.")

    return dataset_df


dataset_df = time_series_processor(dataset_reduction)


def add_target_lags(X, y, timesteps=60):
    X = X.copy()
    y = pd.Series(y, index=X.index)

    lag_features = pd.concat(
        {f"target_lag_{lag}": y.shift(lag) for lag in range(1, timesteps + 1)},
        axis=1,
    )

    X = X.join(lag_features)
    valid = lag_features.notna().all(axis=1)

    return X.loc[valid], y.loc[valid]

## Feature Selection

In [ ]:
X = dataset_df.drop(columns=[target_column]).copy()
y = dataset_df[target_column]

print("The features and the target have been sucessfully defined and the dataset has been split into features (X) and target (y).")

## Data Cleaning

In [ ]:
def data_cleaning(X, y):
    if len(X) != len(y):
        raise ValueError("X and y must contain the same number of rows.")

    X = X.copy()
    cleaning_report = []

    def log_change(step, *, column=None, change=None, rows=None, details=None):
        cleaning_report.append(
            {
                "step": step,
                "column": column,
                "change": change,
                "rows_affected": rows,
                "details": details,
            }
        )

    # Convert array-like y to an indexed Series, or align Series inputs.
    if isinstance(y, pd.Series):
        y = y.copy()
        if not X.index.equals(y.index):
            original_x_rows, original_y_rows = len(X), len(y)
            X, y = X.align(y, join="inner", axis=0)
            dropped = (original_x_rows - len(X)) + (original_y_rows - len(y))
            if dropped:
                log_change(
                    "align_X_y",
                    change="Removed rows without matching indexes",
                    rows=dropped,
                )
    else:
        y = pd.Series(np.asarray(y).ravel(), index=X.index, name="target")

    # Clean target labels and identify valid target rows.
    if (
        pd.api.types.is_object_dtype(y)
        or pd.api.types.is_string_dtype(y)
        or pd.api.types.is_bool_dtype(y)
        or isinstance(y.dtype, pd.CategoricalDtype)
    ):
        cleaned_y = y.astype("string").str.strip()
        cleaned_y = cleaned_y.replace(
            ["", "nan", "NA", "N/A", "None", "null"],
            pd.NA,
        )

        changed = y.astype("string").fillna("<NA>").ne(cleaned_y.fillna("<NA>"))
        if changed.any():
            log_change(
                "clean_target_labels",
                column=y.name,
                change="Trimmed labels or converted configured missing tokens to NA",
                rows=int(changed.sum()),
            )

        y = cleaned_y
        valid_target = y.notna()

    elif pd.api.types.is_numeric_dtype(y):
        cleaned_y = pd.to_numeric(y, errors="coerce")
        valid_target = cleaned_y.notna() & np.isfinite(cleaned_y)

        newly_invalid = cleaned_y.isna() & y.notna()
        if newly_invalid.any():
            log_change(
                "coerce_target",
                column=y.name,
                change="Converted invalid numeric target values to NA",
                rows=int(newly_invalid.sum()),
            )

        infinite_target = cleaned_y.notna() & ~np.isfinite(cleaned_y)
        if infinite_target.any():
            log_change(
                "clean_target_infinities",
                column=y.name,
                change="Removed rows with infinite target values",
                rows=int(infinite_target.sum()),
            )

        y = cleaned_y
    else:
        raise ValueError(f"Unsupported target dtype: {y.dtype}")

    # Replace infinite numeric feature values with NA.
    for column in X.select_dtypes(include=np.number).columns:
        infinite = X[column].isin([np.inf, -np.inf])
        if infinite.any():
            X.loc[infinite, column] = np.nan
            log_change(
                "clean_feature_infinities",
                column=column,
                change="Replaced positive/negative infinity with NA",
                rows=int(infinite.sum()),
            )

    # Apply target-validity mask to both X and y.
    invalid_target = ~valid_target
    if invalid_target.any():
        log_change(
            "remove_invalid_target_rows",
            column=y.name,
            change="Removed rows with missing or invalid targets",
            rows=int(invalid_target.sum()),
        )
        X = X.loc[valid_target].copy()
        y = y.loc[valid_target].copy()

    if cleaning_report:
        cleaning_report = pd.DataFrame(cleaning_report)
        print("Data cleaning operations performed on this dataset:")
        display(cleaning_report)

    else:
        cleaning_report = pd.DataFrame(
            columns=[
                "The cleaning operation is empty. All rows were preserved in their original format."
            ]
        )
        print(
            "No data cleaning operations were required for this dataset. All rows were preserved in their original format."
        )

    return X, y, cleaning_report


X, y, cleaning_report = data_cleaning(X, y)

saved_paths_cleaning = []
csv_cleaning_path = MANIFEST_DIR / "cleaning_report.csv"
cleaning_report.to_csv(csv_cleaning_path, index=True)
saved_paths_cleaning.append(csv_cleaning_path)
csv_cd_path = MANIFEST_DIR / "dataset_cleaned.csv"
dataset_cleaned = pd.concat([X, y], axis=1)
dataset_cleaned.to_csv(csv_cd_path, index=True)
saved_paths_cleaning.append(csv_cd_path)
print("\nDataset cleaning checkpoint saved successfully. Generated artifacts:")
for path in saved_paths_cleaning:
    print(f"  • {Path(path).name}")

## Training/Test Split

In [ ]:
if TIME_SERIES:
    X_lagged, y_lagged = add_target_lags(X, y, timesteps=60)


X_train, X_test, y_train, y_test = train_test_split(
    X if CROSS_SECTIONAL else X_lagged,
    y if CROSS_SECTIONAL else y_lagged,
    test_size=TEST_SIZE,
    shuffle=SHUFFLE,
    stratify=y if CLASSIFICATION_ANALYSIS and SHUFFLE else None,
    random_state=RANDOM_STATE if SHUFFLE else None,
)

print("The dataset has been split into training and test components:")
print(f"Train size: {X_train.shape} | " f"Test size: {X_test.shape}")


# Create manifest tracking dataframes using the sample indices
train_manifest = pd.DataFrame({"split_assignment": "train"}, index=X_train.index)
test_manifest = pd.DataFrame({"split_assignment": "test"}, index=X_test.index)

# Combine them into a single, comprehensive manifest table
split_manifest_df = pd.concat([train_manifest, test_manifest]).sort_index()

# Export directly to your manifest directory using the / operator
saved_paths_split = MANIFEST_DIR / "split_manifest.csv"
split_manifest_df.to_csv(saved_paths_split, index=True)

print("\nSplit manifest checkpoint saved successfully. Generated artifacts:")
print(f"  • {saved_paths_split.name}")

## Preprocessor Pipelines

In [ ]:
_preprocessors_nlp_checkpointed = False
_preprocessors_std_checkpointed = False
_preprocessors_oe_checkpointed = False
_preprocessors_ohe_checkpointed = False


def data_preprocessors(
    X,
    y,
    nlp_column,
    model_name,
    date_columns,
    selected_date_column,
    *,
    min_category_frequency=None,
):

    global _preprocessors_nlp_checkpointed
    global _preprocessors_std_checkpointed
    global _preprocessors_oe_checkpointed
    global _preprocessors_ohe_checkpointed

    preprocessor = None

    if len(X) != len(y):
        raise ValueError("X and y must contain the same number of rows.")

    numeric_transformers = []
    temporal_transformers = []
    categorical_transformers = []

    if CLASSIFICATION_ANALYSIS and NATURAL_LANGUAGE_PROCESSING:

        n_samples = len(X)
        stop_words = set(stopwords.words("english"))
        stop_words.discard("not")

        # Vocabulary scales with dataset size
        max_features = min(max(5_000, int(np.sqrt(n_samples) * 15)), 20_000)

        # Rare-term filtering scales with dataset size
        min_df = max(2, min(int(n_samples * 0.00002), 20))

        transformers = TfidfVectorizer(
            lowercase=True,
            stop_words=list(stop_words),
            ngram_range=(1, 2) if n_samples <= 100_000 else (1, 1),
            min_df=min_df,
            max_df=0.95,
            max_features=max_features,
            sublinear_tf=True,
            dtype=np.float32,
        )

        preprocessor = ColumnTransformer(
            transformers=[("nlp_transformer", transformers, nlp_column)],
            remainder="drop",
            sparse_threshold=1.0,
            verbose_feature_names_out=False,
        )

        if not _preprocessors_nlp_checkpointed:
            saved_paths_preprocessor = []
            saved_paths_preprocessor.extend(
                joblib.dump(preprocessor, CHECKPOINT_DIR / "preprocessor_nlp.joblib")
            )
            print(
                f"Natural Language preprocessor checkpoint successfully saved. Generated artifacts:"
            )
            for path in saved_paths_preprocessor:
                print(f"  • {Path(path).name}\n")
            _preprocessors_nlp_checkpointed = True
    else:

        if not isinstance(X, pd.DataFrame):
            raise TypeError("X must be a pandas DataFrame.")

        # Select numeric features.
        numeric_columns = X.select_dtypes(include=np.number).columns.tolist()

        # Select categorical features, excluding known date columns.
        categorical_columns = (
            X.select_dtypes(include=["object", "string", "category"])
            .drop(columns=date_columns, errors="ignore")
            .columns.tolist()
        )

        # Numeric Pipeline
        # ------------------

        if numeric_columns:
            numeric_steps = [
                ("imputer", SimpleImputer(strategy="median")),
            ]

            if model_name not in {"XGBoost", "Random Forest", "Decision Tree"}:
                numeric_steps.append(("scaler", StandardScaler()))

            numeric_pipeline = Pipeline(numeric_steps)

            numeric_transformers.append(("numeric", numeric_pipeline, numeric_columns))

        # Date Pipeline
        # ------------------
        if date_columns:

            if TIME_SERIES:
                date_steps = FunctionTransformer(
                    lambda X: pd.DataFrame(
                        {
                            "hour_sin": np.sin(
                                2
                                * np.pi
                                * (
                                    X[selected_date_column].dt.hour
                                    + X[selected_date_column].dt.minute / 60
                                )
                                / 24
                            ),
                            "hour_cos": np.cos(
                                2
                                * np.pi
                                * (
                                    X[selected_date_column].dt.hour
                                    + X[selected_date_column].dt.minute / 60
                                )
                                / 24
                            ),
                            "day_of_week_sin": np.sin(
                                2 * np.pi * X[selected_date_column].dt.dayofweek / 7
                            ),
                            "day_of_week_cos": np.cos(
                                2 * np.pi * X[selected_date_column].dt.dayofweek / 7
                            ),
                            "day_of_year_sin": np.sin(
                                2
                                * np.pi
                                * X[selected_date_column].dt.dayofyear
                                / 365.25
                            ),
                            "day_of_year_cos": np.cos(
                                2
                                * np.pi
                                * X[selected_date_column].dt.dayofyear
                                / 365.25
                            ),
                            "year": X[selected_date_column].dt.year,
                        },
                        index=X.index,
                    ),
                    validate=False,
                )
            else:

                date_steps = FunctionTransformer(
                    lambda X: pd.concat(
                        [
                            pd.DataFrame(
                                {
                                    f"{col}_year": pd.to_datetime(
                                        X[col], errors="coerce"
                                    ).dt.year,
                                    f"{col}_month": pd.to_datetime(
                                        X[col], errors="coerce"
                                    ).dt.month,
                                    f"{col}_day": pd.to_datetime(
                                        X[col], errors="coerce"
                                    ).dt.day,
                                },
                                index=X.index,
                            )
                            for col in X.columns
                        ],
                        axis=1,
                    ),
                    validate=False,
                )

            date_pipeline = Pipeline(
                [
                    ("date_steps", date_steps),
                    ("imputer", SimpleImputer(strategy="median")),
                ]
            )

            temporal_transformers.append(("date", date_pipeline, date_columns))

        transformers = numeric_transformers + temporal_transformers

        preprocessor = ColumnTransformer(
            transformers=transformers,
            remainder="drop",
            sparse_threshold=1.0,
            verbose_feature_names_out=False,
        )
        if not _preprocessors_std_checkpointed and not categorical_columns:
            saved_paths_preprocessor = []
            saved_paths_preprocessor.extend(
                joblib.dump(preprocessor, CHECKPOINT_DIR / "preprocessor_std.joblib")
            )
            print(
                f"The numerical and/or temporal preprocessor checkpoint successfully saved. Generated artifacts:"
            )
            for path in saved_paths_preprocessor:
                print(f"  • {Path(path).name}\n")
            _preprocessors_std_checkpointed = True

        #  Categorical Pipelines
        # ---------------------------
        #  One-hot encoding pipeline

        if model_name in {
            "Multiple Linear Regression",
            "Polynomial Regression",
            "Support Vector Regression",
            "Support Vector Machine",
            "K-Nearest Neighbors",
            "Logistic Regression",
            "XGBoost",
        }:
            if categorical_columns:
                categorical_pipeline = Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(strategy="most_frequent"),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                sparse_output=True,
                                min_frequency=min_category_frequency,
                                dtype=np.float32,
                            ),
                        ),
                    ]
                )

                categorical_transformers.append(
                    (
                        "categorical",
                        categorical_pipeline,
                        categorical_columns,
                    )
                )

                transformers = (
                    numeric_transformers
                    + temporal_transformers
                    + categorical_transformers
                )

                # One-hot encoding preprocessor

                preprocessor = ColumnTransformer(
                    transformers=transformers,
                    remainder="drop",
                    sparse_threshold=1.0,
                    verbose_feature_names_out=False,
                )

                if not _preprocessors_ohe_checkpointed:
                    saved_paths_preprocessor = []
                    saved_paths_preprocessor.extend(
                        joblib.dump(
                            preprocessor, CHECKPOINT_DIR / "preprocessor_ohe.joblib"
                        )
                    )
                    print(
                        f"One-hot encoding preprocessor checkpoint successfully saved. Generated artifacts:"
                    )
                    for path in saved_paths_preprocessor:
                        print(f"  • {Path(path).name}\n")
                    _preprocessors_ohe_checkpointed = True

        #  Ordinal encoding pipeline

        elif model_name in {
            "Random Forest",
            "Decision Tree",
            "Naive Bayes",
        }:
            if categorical_columns:
                categorical_pipeline = Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(strategy="most_frequent"),
                        ),
                        (
                            "encoder",
                            OrdinalEncoder(
                                handle_unknown="use_encoded_value",
                                unknown_value=-1,
                                dtype=np.float32,
                            ),
                        ),
                    ]
                )

                categorical_transformers.append(
                    (
                        "categorical",
                        categorical_pipeline,
                        categorical_columns,
                    )
                )

                transformers = (
                    numeric_transformers
                    + temporal_transformers
                    + categorical_transformers
                )

                # Ordinal encoding preprocessor
                preprocessor = ColumnTransformer(
                    transformers=transformers,
                    remainder="drop",
                    sparse_threshold=1.0,
                    verbose_feature_names_out=False,
                )

                if not _preprocessors_oe_checkpointed:
                    saved_paths_preprocessor = []
                    saved_paths_preprocessor.extend(
                        joblib.dump(
                            preprocessor, CHECKPOINT_DIR / "preprocessor_oe.joblib"
                        )
                    )
                    print(
                        f"Ordinal encoding preprocessor checkpoint successfully saved. Generated artifacts:"
                    )
                    for path in saved_paths_preprocessor:
                        print(f"  • {Path(path).name}\n")
                    _preprocessors_oe_checkpointed = True

        else:
            raise ValueError(f"The preprocessor for {model_name} wasn't found.")

    return preprocessor


def preprocessor_selection(model_name):

    preprocessor = data_preprocessors(
        X_train,
        y_train,
        nlp_column,
        model_name,
        date_columns,
        selected_date_column,
        min_category_frequency=10,
    )

    return preprocessor


print(
    "Preprocessor(s) have been fully configured and initialized for downstream modeling."
)

## Model Selection

In [ ]:
def model_selection(
    X_train,
    y_train,
    *,
    cv=2,
):

    if len(X_train) != len(y_train):
        raise ValueError("X_train and y_train must contain the same number of rows.")

    if TIME_SERIES:
        folds = TimeSeriesSplit(
            n_splits=cv,
            gap=0,
        )

    elif CLASSIFICATION_ANALYSIS:
        folds = StratifiedKFold(
            n_splits=cv,
            shuffle=SHUFFLE,
            random_state=(RANDOM_STATE if SHUFFLE else None),
        )

    else:
        folds = KFold(
            n_splits=cv,
            shuffle=SHUFFLE,
            random_state=(RANDOM_STATE if SHUFFLE else None),
        )

    if REGRESSION_ANALYSIS:
        models = {
            # LinearRegression with all features
            "Multiple Linear Regression": Pipeline(
                [
                    (
                        "preprocessor",
                        preprocessor_selection("Multiple Linear Regression"),
                    ),
                    (
                        "estimator",
                        LinearRegression(
                            fit_intercept=True,
                            copy_X=True,
                            tol=1e-4,
                            n_jobs=-1,
                            positive=False,
                        ),
                    ),
                ]
            ),
            # PolynomialFeatures followed by LinearRegression
            "Polynomial Regression": Pipeline(
                [
                    ("preprocessor", preprocessor_selection("Polynomial Regression")),
                    (
                        "feature_selection",
                        SelectKBest(
                            score_func=f_regression,
                            k=20,
                        ),
                    ),
                    (
                        "polynomial_features",
                        PolynomialFeatures(
                            degree=2,
                            interaction_only=True,
                            include_bias=False,
                            order="C",
                        ),
                    ),
                    (
                        "estimator",
                        LinearRegression(
                            fit_intercept=True,
                            copy_X=False,
                        ),
                    ),
                ]
            ),
            "Decision Tree": Pipeline(
                [
                    ("preprocessor", preprocessor_selection("Decision Tree")),
                    (
                        "estimator",
                        DecisionTreeRegressor(
                            max_features="sqrt",  # makes processing fast for encoded datasets,
                            random_state=RANDOM_STATE,
                        ),
                    ),
                ]
            ),
            "Random Forest": Pipeline(
                [
                    ("preprocessor", preprocessor_selection("Random Forest")),
                    (
                        "estimator",
                        RandomForestRegressor(
                            max_features="sqrt",
                            n_jobs=-1,
                            random_state=RANDOM_STATE,
                        ),
                    ),
                ]
            ),
            "XGBoost": Pipeline(
                [
                    ("preprocessor", preprocessor_selection("XGBoost")),
                    (
                        "estimator",
                        XGBRegressor(
                            n_jobs=-1,
                            random_state=RANDOM_STATE,
                        ),
                    ),
                ]
            ),
        }

        if X_train.shape[0] <= 50_000:
            models["Support Vector Regression"] = Pipeline(
                [
                    (
                        "preprocessor",
                        preprocessor_selection("Support Vector Regression"),
                    ),
                    (
                        "feature_selection",
                        SelectKBest(
                            score_func=f_regression,
                            k=50,
                        ),
                    ),
                    (
                        "estimator",
                        SVR(
                            kernel="rbf",
                            tol=1e-2,
                            cache_size=500,
                            max_iter=3_000,
                        ),
                    ),
                ]
            )

        else:
            print(
                "Support Vector Regression excluded: "
                f"{X_train.shape[0]:,} training rows exceed "
                f"the 50,000-row limit established for this method."
            )

        results = []
        print("\nComparing performance of candidate models:\n")
        for model_name, model in tqdm(
            models.items(),
            total=len(models),
            desc="\nEvaluating models\n",
            bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]",
        ):

            scores = cross_validate(
                model,
                X_train,
                y_train,
                scoring={
                    "r2": "r2",
                    "rmse": "neg_root_mean_squared_error",
                    "mae": "neg_mean_absolute_error",
                },
                cv=folds,
                n_jobs=1,
                verbose=1,
                error_score="raise",
            )

            results.append(
                {
                    "model": model_name,
                    "mean_cv_mae": -(scores["test_mae"].mean()),
                    "std_cv_mae": scores["test_mae"].std(),
                    "mean_cv_rmse": -(scores["test_rmse"].mean()),
                    "std_cv_rmse": scores["test_rmse"].std(),
                    "mean_cv_r2": scores["test_r2"].mean(),
                    "std_cv_r2": scores["test_r2"].std(),
                }
            )

        results = pd.DataFrame(results)
        default_results = (
            results.loc[
                (results["mean_cv_r2"] > 0.1) & (results["mean_cv_r2"] <= 0.995)
            ]
            .sort_values(by=["mean_cv_mae", "mean_cv_r2"], ascending=[True, False])
            .reset_index(drop=True)
        )

        if default_results.empty:
            raise ValueError(
                "No regression results were produced that met the criteria standards. This may be caused by a weak predictive signal or target unpredictability, which renders this dataset unsuitable for the task."
            )

    elif CLASSIFICATION_ANALYSIS:

        models = {
            "Logistic Regression": Pipeline(
                [
                    ("preprocessor", preprocessor_selection("Logistic Regression")),
                    (
                        "estimator",
                        LogisticRegression(
                            solver="saga",
                            max_iter=1_000,
                            random_state=RANDOM_STATE,
                        ),
                    ),
                ]
            ),
            "Naive Bayes": Pipeline(
                [
                    ("preprocessor", preprocessor_selection("Naive Bayes")),
                    (
                        "estimator",
                        MultinomialNB() if nlp_flag else GaussianNB(),
                    ),
                ]
            ),
        }
        if not NATURAL_LANGUAGE_PROCESSING:
            models.update(
                {
                    "Decision Tree": Pipeline(
                        [
                            ("preprocessor", preprocessor_selection("Decision Tree")),
                            (
                                "estimator",
                                DecisionTreeClassifier(
                                    max_features="sqrt",
                                    random_state=RANDOM_STATE,
                                ),
                            ),
                        ]
                    ),
                    "Random Forest": Pipeline(
                        [
                            ("preprocessor", preprocessor_selection("Random Forest")),
                            (
                                "estimator",
                                RandomForestClassifier(
                                    max_features="sqrt",
                                    n_jobs=-1,
                                    random_state=RANDOM_STATE,
                                ),
                            ),
                        ]
                    ),
                    "XGBoost": Pipeline(
                        [
                            ("preprocessor", preprocessor_selection("XGBoost")),
                            (
                                "estimator",
                                XGBClassifier(
                                    n_jobs=-1,
                                    random_state=RANDOM_STATE,
                                ),
                            ),
                        ]
                    ),
                }
            )

            # Add KNN and Support Vector Machine only when the training dataset is sufficiently small.
            if X_train.shape[0] <= 100_000:
                models["K-Nearest Neighbors"] = Pipeline(
                    [
                        ("preprocessor", preprocessor_selection("K-Nearest Neighbors")),
                        (
                            "feature_selection",
                            SelectKBest(
                                score_func=f_classif,
                                k=50,
                            ),
                        ),
                        (
                            "estimator",
                            KNeighborsClassifier(
                                n_neighbors=15,
                                weights="distance",
                                algorithm="brute",
                                metric="euclidean",
                                n_jobs=-1,
                            ),
                        ),
                    ]
                )
                if X_train.shape[0] <= 50_000:
                    models["Support Vector Machine"] = Pipeline(
                        [
                            (
                                "preprocessor",
                                preprocessor_selection("Support Vector Machine"),
                            ),
                            (
                                "feature_selection",
                                SelectKBest(
                                    score_func=f_classif,
                                    k=50,
                                ),
                            ),
                            (
                                "estimator",
                                SVC(
                                    kernel="rbf",
                                    tol=1e-2,
                                    cache_size=500,
                                    max_iter=3_000,
                                    probability=False,
                                    random_state=RANDOM_STATE,
                                ),
                            ),
                        ]
                    )

                else:
                    print(
                        "Support Vector Machine excluded: "
                        f"{X_train.shape[0]:,} training rows exceed "
                        f"the 50,000-row limit established for this method."
                    )

            else:
                print(
                    "K-Nearest Neighbors and Support Vector Machine excluded: "
                    f"{X_train.shape[0]:,} training rows exceed "
                    f"the 100,000-row and 50,000-row limits established for these methods, respectively."
                )

        results = []
        print("\nComparing performance of candidate models:\n")
        for model_name, model in tqdm(
            models.items(),
            total=len(models),
            desc="Evaluating models",
            bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]",
        ):

            scores = cross_validate(
                model,
                X_train,
                y_train,
                scoring={
                    "accuracy": "accuracy",
                    "balanced_accuracy": "balanced_accuracy",
                    "precision": "precision_macro",
                    "recall": "recall_macro",
                    "f1": "f1_macro",
                },
                cv=folds,
                n_jobs=1,
                verbose=1,
                error_score="raise",
            )

            results.append(
                {
                    "model": model_name,
                    "mean_cv_f1": scores["test_f1"].mean(),
                    "std_cv_f1": scores["test_f1"].std(),
                    "mean_cv_balanced_accuracy": scores["test_balanced_accuracy"].mean(),
                    "std_cv_balanced_accuracy": scores["test_balanced_accuracy"].std(),
                    "mean_cv_accuracy": scores["test_accuracy"].mean(),
                    "std_cv_accuracy": scores["test_accuracy"].std(),
                    "mean_cv_precision": scores["test_precision"].mean(),
                    "std_cv_precision": scores["test_precision"].std(),
                    "mean_cv_recall": scores["test_recall"].mean(),
                    "std_cv_recall": scores["test_recall"].std(),
                }
            )

        results = pd.DataFrame(results)
        default_results = (
            results.loc[results["mean_cv_f1"] <= 0.995]
            .sort_values(
                by=["mean_cv_f1", "mean_cv_balanced_accuracy"],
                ascending=[False, False],
            )
            .reset_index(drop=True)
        )

        if default_results.empty:
            raise ValueError(
                "No classification results were produced that met the criteria standards. This may be caused by a weak predictive signal or target unpredictability, which renders this dataset unsuitable for the task."
            )

    selected_model_name = default_results.loc[0, "model"]
    selected_model = clone(models[selected_model_name])

    selected_model.fit(X_train, y_train)

    print("\nBest model:", selected_model_name)
    display(default_results)
    print("Selected model:", selected_model_name)

    return selected_model, default_results, selected_model_name


selected_model, default_results, selected_model_name = model_selection(
    X_train,
    y_train,
    cv=2,
)

saved_paths_deft = []
csv_deft_path = METRICS_DIR / "model_comparison.csv"
default_results.to_csv(csv_deft_path, index=False)
saved_paths_deft.append(csv_deft_path)
saved_paths_deft.extend(
    joblib.dump(selected_model, CHECKPOINT_DIR / "model_default_profile.joblib")
)
print(
    f"\nModel selection checkpoint successfully saved. Henceforth, the selected model will be referred to as default model. Generated artifacts:"
)
for path in saved_paths_deft:
    print(f"  • {Path(path).name}")

## Model Optimization

In [ ]:
def model_hyperparameter_tuning(selected_model_name, cv=5):

    if REGRESSION_ANALYSIS:
        optimization_scoring = "neg_mean_absolute_error"

        if selected_model_name == "XGBoost":
            parameters = {
                "estimator__n_estimators": [100, 300, 500],
                "estimator__learning_rate": [0.01, 0.03, 0.05, 0.1],
                "estimator__max_depth": [3, 5, 8],
                "estimator__min_child_weight": [1, 3, 5],
                "estimator__subsample": [0.6, 0.8, 1.0],
                "estimator__colsample_bytree": [0.6, 0.8, 1.0],
            }

        elif selected_model_name == "Random Forest":
            parameters = {
                "estimator__n_estimators": [50, 100, 200],
                "estimator__max_depth": [10, 20, None],
                "estimator__min_samples_leaf": [5, 25, 50],
                "estimator__min_samples_split": [10, 50, 100],
                "estimator__max_features": ["sqrt", 0.5, 0.8],
                "estimator__max_samples": [0.5, 0.8, 1],
            }

        elif selected_model_name == "Decision Tree":
            parameters = {
                "estimator__splitter": ["best", "random"],
                "estimator__max_depth": [10, 20, None],
                "estimator__min_samples_split": [10, 50, 100],
                "estimator__min_samples_leaf": [5, 25, 50],
                "estimator__max_features": ["sqrt", 0.5, None],
            }

        elif selected_model_name == "Support Vector Regression":
            parameters = {
                "feature_selection__k": [20, 50, 100],
                "estimator__C": [0.1, 1.0, 10.0],
                "estimator__epsilon": [0.1, 0.3, 0.5],
                "estimator__gamma": ["scale", 0.001, 0.01],
                "estimator__tol": [0.01, 0.05, 0.1],
            }

        elif selected_model_name == "Polynomial Regression":
            parameters = {
                "feature_selection__k": [
                    20,
                    30,
                    50,
                ],
                "polynomial_features__degree": [1, 2],
                "polynomial_features__interaction_only": [True, False],
                "estimator__fit_intercept": [True, False],
                "estimator__tol": [1e-3, 1e-4, 1e-6],
            }

        elif selected_model_name == "Multiple Linear Regression":
            parameters = {
                "estimator__fit_intercept": [True, False],
                "estimator__tol": [1e-3, 1e-4, 1e-6],
            }

        else:
            raise ValueError("No model was found")

    elif CLASSIFICATION_ANALYSIS:
        optimization_scoring = "f1_macro"

        if selected_model_name == "XGBoost":
            parameters = {
                "estimator__n_estimators": [100, 300, 500],
                "estimator__learning_rate": [0.01, 0.03, 0.05, 0.1],
                "estimator__max_depth": [3, 5, 8],
                "estimator__min_child_weight": [1, 3, 5],
                "estimator__subsample": [0.6, 0.8, 1.0],
                "estimator__colsample_bytree": [0.6, 0.8, 1.0],
            }

        elif selected_model_name == "Random Forest":
            parameters = {
                "estimator__n_estimators": [50, 100, 200],
                "estimator__max_depth": [10, 20, None],
                "estimator__min_samples_leaf": [1, 5, 20],
                "estimator__min_samples_split": [2, 10, 50],
                "estimator__max_features": ["sqrt", 0.5, 0.8],
                "estimator__max_samples": [0.5, 0.8, 1],
                "estimator__class_weight": [None, "balanced", "balanced_subsample"],
            }

        elif selected_model_name == "Decision Tree":
            parameters = {
                "estimator__criterion": ["gini", "entropy", "log_loss"],
                "estimator__splitter": ["best", "random"],
                "estimator__max_depth": [10, 20, None],
                "estimator__min_samples_split": [2, 10, 50],
                "estimator__min_samples_leaf": [1, 5, 20],
                "estimator__max_features": ["sqrt", 0.5, None],
                "estimator__class_weight": [None, "balanced"],
            }

        elif selected_model_name == "Support Vector Machine":
            parameters = {
                "feature_selection__k": [20, 50, 100],
                "estimator__C": [0.1, 1.0, 10.0],
                "estimator__class_weight": [None, "balanced"],
                "estimator__gamma": ["scale", 0.001, 0.01],
                "estimator__tol": [0.01, 0.05, 0.1],
            }

        elif selected_model_name == "Naive Bayes":
            parameters = {
                "estimator__var_smoothing": [1e-11, 1e-9, 1e-7],
            }

        elif selected_model_name == "K-Nearest Neighbors":
            parameters = {
                "feature_selection__k": [20, 50, 100],
                "estimator__n_neighbors": [5, 15, 31],
                "estimator__weights": ["uniform", "distance"],
                "estimator__metric": ["euclidean", "manhattan"],
            }

        elif selected_model_name == "Logistic Regression":
            parameters = {
                "estimator__C": [0.1, 1.0, 10.0],
                "estimator__tol": [1e-3, 1e-4, 1e-5],
                "estimator__class_weight": [None, "balanced"],
                "estimator__fit_intercept": [True, False],
            }

        else:
            raise ValueError("No model was found")
    else:
        raise ValueError("No project type was found")

    if selected_model_name:
        if TIME_SERIES:
            folds = TimeSeriesSplit(
                n_splits=cv,
                gap=0,
            )

            search_estimator = clone(selected_model).set_params(
                preprocessor=preprocessor_selection(selected_model_name)
            )

            estimator_optm = RandomizedSearchCV(
                estimator=search_estimator,
                param_distributions=parameters,
                n_iter=25,
                scoring=optimization_scoring,
                cv=folds,
                refit=True,
                n_jobs=1,
                random_state=RANDOM_STATE,
                verbose=1,
                error_score="raise",
                return_train_score=True,
            )
        else:

            if CLASSIFICATION_ANALYSIS:
                folds = StratifiedKFold(
                    n_splits=cv,
                    shuffle=SHUFFLE,
                    random_state=(RANDOM_STATE if SHUFFLE else None),
                )

            else:
                folds = KFold(
                    n_splits=cv,
                    shuffle=SHUFFLE,
                    random_state=(RANDOM_STATE if SHUFFLE else None),
                )

            search_estimator = clone(selected_model).set_params(
                preprocessor=preprocessor_selection(selected_model_name)
            )

            estimator_optm = HalvingRandomSearchCV(
                estimator=search_estimator,
                param_distributions=parameters,
                min_resources="exhaust",
                max_resources="auto",
                n_candidates=9,
                factor=3,
                resource="n_samples",
                aggressive_elimination=True,
                scoring=optimization_scoring,
                cv=folds,
                refit=True,
                n_jobs=1,
                random_state=RANDOM_STATE,
                verbose=1,
                error_score="raise",
                return_train_score=True,
            )

        print(
            f"Best hyperparameter combination for {selected_model_name} model found and stored."
        )

        return estimator_optm
    else:
        raise ValueError("No model was found")


estimator_optm = model_hyperparameter_tuning(selected_model_name, cv=5)

## Training Model on the Training Set

In [ ]:
print(
    f"Optimizing hyperparameters for {selected_model_name} and fitting the model to the training set:\n"
)
estimator_optm.fit(X_train, y_train)

best_parameters = estimator_optm.best_params_
best_parameters_table = pd.DataFrame(
    best_parameters.items(), columns=["Parameter", "Selected Value"]
)
full_estimator_optm = estimator_optm.best_estimator_.get_params(deep=True)
optmization_cv_results = pd.DataFrame(estimator_optm.cv_results_)

print("\nBest Parameters Found:")
display(best_parameters_table)
print(f"\n{selected_model_name} was fitted to the training set!\n")

saved_paths_optm = []
saved_paths_optm.extend(
    joblib.dump(best_parameters, CHECKPOINT_DIR / "best_parameters_optm.joblib")
)
saved_paths_optm.extend(
    joblib.dump(full_estimator_optm, CHECKPOINT_DIR / "model_optimized_profile.joblib")
)
csv_optm_path = METRICS_DIR / "optimization_cv_results.csv"
optmization_cv_results.to_csv(csv_optm_path, index=False)
saved_paths_optm.append(csv_optm_path)
print(
    f"Model optimization checkpoint successfully saved. Henceforth, the resulting model from this operation will be referred to as the optimized model. Generated artifacts:"
)
for path in saved_paths_optm:
    print(f"  • {Path(path).name}")

## Default and Optimized Model Comparison

In [ ]:
def choose_default_or_optimized(selected_model, estimator_optm, default_results):

    default_model = selected_model
    optimized_model = estimator_optm.best_estimator_

    if REGRESSION_ANALYSIS:

        optimized_mae = -(estimator_optm.best_score_)
        default_mae = float(default_results.loc[0, "mean_cv_mae"])

        final_comparison = pd.DataFrame(
            [
                {
                    "configuration": "Default",
                    "mean_cv_mae": default_mae,
                },
                {
                    "configuration": "Optimized",
                    "mean_cv_mae": optimized_mae,
                },
            ]
        )

        optimized_is_better = optimized_mae < default_mae

        if optimized_is_better:
            final_estimator = optimized_model
            selected_configuration = "optimized"
            selected_parameters = estimator_optm.best_params_
        else:
            final_estimator = default_model
            selected_configuration = "default"
            selected_parameters = default_model.get_params(deep=True)

    elif CLASSIFICATION_ANALYSIS:

        optimized_f1 = estimator_optm.best_score_
        default_f1 = float(default_results.loc[0, "mean_cv_f1"])

        final_comparison = pd.DataFrame(
            [
                {
                    "configuration": "Default",
                    "mean_cv_f1": default_f1,
                },
                {
                    "configuration": "Optimized",
                    "mean_cv_f1": optimized_f1,
                },
            ]
        )

        optimized_is_better = optimized_f1 > default_f1

        if optimized_is_better:
            final_estimator = optimized_model
            selected_configuration = "optimized"
            selected_parameters = estimator_optm.best_params_
        else:
            final_estimator = default_model
            selected_configuration = "default"
            selected_parameters = default_model.get_params(deep=True)

    print("Comparison between default and optimized model versions:")
    display(final_comparison)
    print("Selected configuration:", selected_configuration)

    return (
        final_estimator,
        final_comparison,
        selected_configuration,
        selected_parameters,
    )


(
    final_estimator,
    final_comparison,
    selected_configuration,
    selected_parameters,
) = choose_default_or_optimized(selected_model, estimator_optm, default_results)

## Predicting the Test Set Results

In [ ]:
y_pred = final_estimator.predict(X_test)

print(
    f"The {selected_configuration} model configuration has generated predictions on the test set."
)

## Model Evaluation

In [ ]:
if REGRESSION_ANALYSIS:

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    final_results_table = pd.DataFrame(
        {"Metric": ["MAE", "RMSE", "R²"], "Value": [mae, rmse, r2]}
    )
    print(
        f"Final evaluation metrics for the {selected_configuration} configuration of the {selected_model_name} model:"
    )
    display(final_results_table.style.hide(axis="index"))

elif CLASSIFICATION_ANALYSIS:

    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    accuracy = accuracy_score(y_test, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_test, y_pred, average="macro", zero_division=0)

    final_results_table = pd.DataFrame(
        {
            "Metric": ["F1", "Accuracy", "Balanced Accuracy", "Precision", "Recall"],
            "Value": [f1, accuracy, balanced_accuracy, precision, recall],
        }
    )
    print(
        f"Final evaluation metrics for the {selected_configuration} configuration of the {selected_model_name} model:"
    )
    display(final_results_table.style.hide(axis="index"))

else:
    print("No model was detected")


saved_paths_results = METRICS_DIR / "final_evaluation_results.csv"
final_results_table.to_csv(saved_paths_results, index=False)

print("\nFinal evaluation results saved successfully. Generated artifacts:")
print(f"  • {saved_paths_results.name}")

## Diagnostics

In [ ]:
if REGRESSION_ANALYSIS:
    residuals = np.asarray(y_test) - np.asarray(y_pred)
    predicted = np.asarray(y_pred)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1. Actual vs predicted
    axes[0].scatter(y_test, y_pred, alpha=0.5)
    axes[0].set_xlabel("Actual values")
    axes[0].set_ylabel("Predicted values")
    axes[0].set_title("Actual vs predicted")

    # 2. Residuals vs predicted
    axes[1].scatter(predicted, residuals, alpha=0.5)
    axes[1].axhline(0, linestyle="--")
    axes[1].set_xlabel("Predicted values")
    axes[1].set_ylabel("Residuals")
    axes[1].set_title("Residuals vs predicted")

    # 3. Residual distribution
    axes[2].hist(residuals, bins=30)
    axes[2].set_xlabel("Residual")
    axes[2].set_ylabel("Frequency")
    axes[2].set_title("Residual distribution")

    plt.tight_layout()
    fig.savefig(
        FIGURE_DIR / "prediction_and_residuals.png", dpi=160, bbox_inches="tight"
    )
    plt.show()
    plt.close(fig)

elif CLASSIFICATION_ANALYSIS:

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm)

    disp.plot()
    plt.title("Confusion Matrix")
    plt.savefig(FIGURE_DIR / "confusion_matrix.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()

    # Classification Report (per-class metrics)
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

    per_class_metrics = pd.DataFrame(report).T

    display(per_class_metrics)

    # ROC-AUC and PR_AUC

    y_proba = final_estimator.predict_proba(X_test)

    n_classes = len(final_estimator.classes_)

    if n_classes == 2:

        # Binary classification
        y_proba_binary = y_proba[:, 1]

        roc_auc = roc_auc_score(y_test, y_proba_binary)

        pr_auc = average_precision_score(y_test, y_proba_binary)

    else:

        # Multiclass classification
        roc_auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="weighted")

        pr_auc = average_precision_score(y_test, y_proba, average="weighted")

    print("\nROC-AUC:", roc_auc)
    print("PR-AUC:", pr_auc)

## Experiment Summary

In [ ]:
def result_analysis_summary(project_type, **kwargs):

    if project_type == "Regression":
        r2 = kwargs["r2"]
        mae = kwargs["mae"]
        rmse = kwargs["rmse"]
        residuals = kwargs["residuals"]

        # Evaluation statement
        if r2 >= 0.999 and mae <= 1 and rmse <= 1:
            evaluation_statement = f"The perfect evaluation metrics obtained are highly symptomatic of an underlying systemic issue rather than an optimal model. This distortion is likely driven by dataset corruption, data leakage, extreme overfitting or a deterministic target-feature relationship (such as a direct mathematical conversion). R² = {r2:.6f}, MAE = {mae:.6f}, RMSE = {rmse:.6f}"
            diagnostic_statement = "Further analysis was not conducted due to the issues reported in the previous statement."
            statistical_statement = "Further analysis was not conducted due to the issues reported in the previous statement."
        else:

            if r2 >= 0.90 and mae <= 5 and rmse <= 7:
                evaluation_statement = f"The regression model demonstrated excellent overall performance. The high R² ({r2:.6f}) indicates that the model was able to explain a very large proportion of the variability in the target variable. The low MAE ({mae:.6f}) indicates that predictions were, on average, very close to the observed values, while the low RMSE ({rmse:.6f}) suggests that large prediction errors were also limited. Overall, the results indicate that the available features contain strong predictive information, allowing the model to capture most of the underlying patterns in the target variable with a high degree of accuracy."

            elif r2 >= 0.75 and mae <= 10 and rmse <= 12:
                evaluation_statement = f"The regression model demonstrated good overall performance. The R² ({r2:.6f}) indicates that the model captured a substantial proportion of the variability in the target variable, while the MAE ({mae:.6f}) indicates relatively small prediction errors. The RMSE ({rmse:.6f}) suggests that larger prediction errors were generally controlled. Overall, the results indicate that the available features contain substantial predictive information, although some variability in the target remains difficult to explain from the available data."

            elif r2 >= 0.50 and mae <= 15 and rmse <= 20:
                evaluation_statement = f"The regression model demonstrated moderate overall performance. The model was able to capture meaningful patterns in the target variable, as reflected by the R² ({r2:.6f}), but a considerable proportion of the variability remains unexplained. The MAE ({mae:.6f}) indicates noticeable prediction errors, while the higher RMSE ({rmse:.6f}) suggests that some larger errors remain. Overall, the results indicate that the available features contain meaningful predictive information, but a considerable proportion of the target variability remains unexplained by the model."

            else:
                evaluation_statement = f"The regression model demonstrated limited predictive performance despite extensive optimization. The relatively low R² ({r2:.6f}) indicates that only a limited proportion of the variability in the target variable could be explained by the available features. The relatively high MAE ({mae:.6f}) indicates substantial average prediction errors, while the higher RMSE ({rmse:.6f}) indicates the presence of larger prediction errors. The results suggest that the main limitation is likely related to the predictive information available in the dataset rather than the model configuration itself. Additional predictive information or more informative features would likely be required to substantially improve performance."

            residual_mean = np.mean(residuals)
            residual_ratio = (
                abs(residual_mean) / float(mae)
                if mae > 0
                else 0.0 if residual_mean == 0 else np.inf
            )

            if residual_mean > 0:
                overall_bias = "underpredicting"
            elif residual_mean < 0:
                overall_bias = "overpredicting"
            else:
                overall_bias = "perfectly balanced"

            # Diagnostic statement
            diagnostic_statement = []
            if residual_ratio == 0:
                diagnostic_statement_rr = f"The model is {overall_bias} overall, with no net systematic overprediction or underprediction. Positive and negative residuals cancel exactly (Relative residual bias:{residual_ratio:.4f})."
            elif residual_ratio <= 0.05:
                diagnostic_statement_rr = f"In general, the model is slightly {overall_bias}, but the systematic bias is negligible relative to the overall error. Residuals are closely balanced around zero (Relative residual bias:{residual_ratio:.4f})."
            elif residual_ratio <= 0.1:
                diagnostic_statement_rr = f"In general, the model is slightly {overall_bias}. The systematic bias is very small relative to the typical prediction error (Relative residual bias:{residual_ratio:.4f})."
            elif residual_ratio <= 0.2:
                diagnostic_statement_rr = f"In general, the model is {overall_bias}. A mild systematic bias is present, although random variation still represents most of the error (Relative residual bias:{residual_ratio:.4f})."
            elif residual_ratio <= 0.3:
                diagnostic_statement_rr = f"In general, the model is {overall_bias}. The residuals show a moderate systematic bias that is large enough to warrant investigation (Relative residual bias:{residual_ratio:.4f})."
            elif residual_ratio <= 0.5:
                diagnostic_statement_rr = f"In general, the model is strongly {overall_bias}. The systematic component of the residual error is substantial relative to the overall prediction error (Relative residual bias:{residual_ratio:.4f})."
            else:
                diagnostic_statement_rr = f"In general, the model is very strongly {overall_bias}. The mean residual represents a large proportion of the overall error, indicating substantial systematic bias (Relative residual bias: {residual_ratio:.4f})."

            diagnostic_statement.append(diagnostic_statement_rr)

            # Blind Spot statement

            dw_stat = durbin_watson(residuals)
            if dw_stat <= 0.5:
                has_blind_spot = True
                blind_spot_reason = f"Residuals show very strong positive autocorrelation, meaning consecutive errors tend to have the same sign. This may indicate an unmodelled sequential or time-related pattern. Plot the errors over time and check that the dates, predictions and test splits are aligned correctly; a lingering trend, repeating pattern or sudden change could explain the result. (Durbin-Watson:{dw_stat:.4f})."
            elif dw_stat <= 1:
                has_blind_spot = True
                blind_spot_reason = f"Residuals show strong positive autocorrelation, with consecutive errors tending to cluster in the same direction. An underlying sequential or time-related pattern may be missing from the model. Plot the errors over time and check that the dates, predictions and test splits are aligned correctly; a lingering trend, repeating pattern or sudden change could explain the result. (Durbin-Watson:{dw_stat:.4f})."
            elif dw_stat <= 1.5:
                has_blind_spot = True
                blind_spot_reason = f"Residuals show moderate positive autocorrelation. Some sequential structure remains unexplained by the model (Durbin-Watson:{dw_stat:.4f})."
            elif dw_stat <= 2.5:
                has_blind_spot = False
                blind_spot_reason = f"Residuals show little evidence of first-order autocorrelation and appear reasonably independent in sequence (Durbin-Watson:{dw_stat:.4f})."
            elif dw_stat <= 3:
                has_blind_spot = True
                blind_spot_reason = f"Residuals show moderate negative autocorrelation, meaning errors tend to alternate signs more than expected under independence (Durbin-Watson:{dw_stat:.4f})."
            elif dw_stat <= 3.5:
                has_blind_spot = True
                blind_spot_reason = f"Residuals show strong negative autocorrelation, with errors tending to alternate signs. This may indicate an unmodelled systematic pattern (Durbin-Watson:{dw_stat:.4f})."
            else:
                has_blind_spot = True
                blind_spot_reason = f"Residuals show very strong negative autocorrelation, with frequent alternation between positive and negative errors. This indicates substantial sequential structure in the residuals. Data may be overprocessed. (Durbin-Watson:{dw_stat:.4f})."

            diagnostic_statement.append(blind_spot_reason)
            diagnostic_statement = " ".join(diagnostic_statement)

            # Statistics statement
            shapiro_stat, p_value_norm = stats.shapiro(residuals)

            # Distance of W from perfect normality
            w_deviation = 1 - shapiro_stat

            if p_value_norm >= 0.05 and w_deviation <= 0.02:
                statistical_statement = f"The statistical distribution of the errors are consistent with a normal distribution. The Shapiro-Wilk test does not provide significant evidence against normality, and the W statistic is very close to 1 (W={shapiro_stat:.4f}, p={p_value_norm})."
            elif p_value_norm >= 0.05 and w_deviation > 0.02:
                statistical_statement = f"The statistical distribution of the errors does not show a statistically significant departure from normality (p={p_value_norm}). However, the W statistic (W={shapiro_stat:.4f}) indicates a minor geometric deviation, likely due to sample characteristics."
            elif p_value_norm < 0.05 and w_deviation <= 0.01:
                statistical_statement = f"The statistical distribution of the errors are approximate to a normal distribution. The Shapiro-Wilk test detects a statistically significant departure from perfect normality, but the W statistic remains very close to 1, suggesting that the deviation may be small in practical terms (W={shapiro_stat:.4f}, p={p_value_norm})."
            elif p_value_norm < 0.05 and w_deviation <= 0.03:
                statistical_statement = f"The statistical distribution of the errors are consistent with a non-normal distribution. The Shapiro-Wilk test rejects normality and the W statistic shows a noticeable departure from 1 (W={shapiro_stat:.4f}, p={p_value_norm}). This may be caused by skewness, heavy tails or outliers."
            else:
                statistical_statement = f"The statistical distribution of the errors are strongly consistent with a non-normal distribution. The Shapiro-Wilk test rejects normality and the W statistic shows a relatively large departure from 1 (W={shapiro_stat:.4f}, p={p_value_norm}). This may be caused by strong curvature, heavy tails, skewness or extreme outliers."

    elif project_type == "Classification":
        f1 = kwargs["f1"]
        balanced_accuracy = kwargs["balanced_accuracy"]
        precision = kwargs["precision"]
        recall = kwargs["recall"]
        roc_auc = kwargs["roc_auc"]
        pr_auc = kwargs["pr_auc"]
        y_test = kwargs["y_test"]
        y_pred = kwargs["y_pred"]
        cm = kwargs["cm"]

        if f1 >= 0.999 or balanced_accuracy >= 0.999:
            evaluation_statement = f"The perfect evaluation metrics obtained are highly symptomatic of an underlying systemic issue rather than an optimal model. This distortion is likely driven by data leakage, extreme overfitting or a deterministic target-feature relationship (such as a direct mathematical conversion). F1 = {f1:.6f}, Balanced Accuracy= {balanced_accuracy:.6f}, Precision = {precision:.6f}, Recall = {recall:.6f}"
            diagnostic_statement = "Further analysis was not conducted due to the issues reported in the previous statement."
            statistical_statement = "Further analysis was not conducted due to the issues reported in the previous statement."

        else:
            if f1 >= 0.9:
                evaluation_statement = (
                    f"The classification model demonstrated excellent overall performance (F1-Score: {f1:.6f}). "
                    f"The high Balanced Accuracy ({balanced_accuracy:.6f}) confirms that the model handles both the majority "
                    f"and minority classes with outstanding precision ({precision:.6f}) and high operational recall ({recall:.6f})."
                )
            elif f1 >= 0.75:
                evaluation_statement = (
                    f"The classification model demonstrated good overall performance (F1-Score: {f1:.6f}). "
                    f"The model shows stable predictive capabilities across classes, though minor classification errors "
                    f"persist. Precision stands at {precision:.6f} with a recall threshold of {recall:.6f}."
                )
            elif f1 >= 0.60:
                evaluation_statement = (
                    f"The classification model demonstrated moderate overall performance (F1-Score: {f1:.6f}). "
                    f"While it successfully extracts meaningful signals to beat a baseline coin-flip, a significant amount "
                    f"of class variance remains misclassified. Additional feature engineering is recommended."
                )
            else:
                evaluation_statement = (
                    f"The classification model demonstrated inadequate predictive performance (F1-Score: {f1:.6f}). "
                    f"The current feature architecture fails to reliably distinguish between target categories, rendering "
                    f"the predictions structurally unsafe for automated deployment or production environments."
                )

            diagnostic_statement = []
            if 0.9 <= roc_auc <= 1:
                diagnostic_statement_roc = f"The model has excellent ability to distinguish positive cases from negative cases. Positive cases are generally assigned higher prediction scores than negative cases (ROC-AUC score: {roc_auc:.4f})."
            elif 0.8 <= roc_auc < 0.9:
                diagnostic_statement_roc = f"The model has good discriminative ability and can generally distinguish positive from negative cases effectively, although some overlap between the two groups remains (ROC-AUC score: {roc_auc:.4f})."
            elif 0.7 <= roc_auc < 0.8:
                diagnostic_statement_roc = f"The model has moderate discriminative ability. It identifies meaningful differences between positive and negative cases, but its predictions show considerable overlap (ROC-AUC score: {roc_auc:.4f})."
            elif 0.6 <= roc_auc < 0.7:
                diagnostic_statement_roc = f"The model has limited discriminative ability. It performs better than random classification, but the separation between positive and negative cases is relatively weak (ROC-AUC score: {roc_auc:.4f})."
            elif 0.5 <= roc_auc < 0.6:
                diagnostic_statement_roc = f"The model has very weak discriminative ability. It performs only slightly better than random ranking, indicating substantial overlap between positive and negative prediction scores (ROC-AUC score: {roc_auc:.4f})."
            elif 0.4 <= roc_auc < 0.5:
                diagnostic_statement_roc = f"The model has weak inverse discriminative ability. Negative cases tend to receive sligthy higher prediction scores than positive cases, although the score is close to random ranking (ROC-AUC score: {roc_auc:.4f})."
            else:
                diagnostic_statement_roc = f"The model demonstrates strong inverse discrimination. Negative cases tend to receive substantially higher prediction scores than positive cases (ROC-AUC score: {roc_auc:.4f})."

            diagnostic_statement.append(diagnostic_statement_roc)

            positive_prevalence = np.mean(y_test)
            pr_auc_lift = (
                pr_auc / positive_prevalence if positive_prevalence > 0 else np.nan
            )

            if pr_auc >= 0.9:
                diagnostic_statement_prauc = f"The model demonstrates exccellent positive-case detection, with a PR-AUC of {pr_auc:.4f}. This corresponds to {pr_auc_lift:.1f}× the positive-class prevalence baseline."
            elif pr_auc >= 0.8:
                diagnostic_statement_prauc = f"The model demonstrates very strong positive-case detection, with a PR-AUC of {pr_auc:.4f}. This corresponds to {pr_auc_lift:.1f}× the positive-class prevalence baseline."
            elif pr_auc >= 0.7:
                diagnostic_statement_prauc = f"The model demonstrates strong positive-case detection, with a PR-AUC of {pr_auc:.4f}. This corresponds to {pr_auc_lift:.1f}× the positive-class prevalence baseline."
            elif pr_auc >= 0.5:
                diagnostic_statement_prauc = f"The model demonstrates moderate positive-case detection, with a PR-AUC of {pr_auc:.4f}. This corresponds to {pr_auc_lift:.1f}× the positive-class prevalence baseline."
            elif pr_auc >= 0.3:
                diagnostic_statement_prauc = f"The model demonstrates limited-to-moderate positive-case detection, with a PR-AUC of {pr_auc:.4f}. This corresponds to {pr_auc_lift:.1f}× the positive-class prevalence baseline."
            else:
                diagnostic_statement_prauc = f"The model demonstrates weak positive-case detection, with a PR-AUC of {pr_auc:.4f}. This corresponds to {pr_auc_lift:.1f}× the positive-class prevalence baseline."

            diagnostic_statement.append(diagnostic_statement_prauc)
            diagnostic_statement = "\n".join(diagnostic_statement)

            n_classes = len(np.unique(y_test))
            statistical_statement = []

            if n_classes == 2:

                # Binary classification
                tn, fp, fn, tp = cm.ravel()

                total = tn + fp + fn + tp

                actual_prevalence = (tp + fn) / total if total > 0 else 0

                predicted_prevalence = (tp + fp) / total if total > 0 else 0
                prediction_bias = predicted_prevalence - actual_prevalence

                if prediction_bias >= 0.1:
                    statistical_statement_pb = f"The model shows strong overprediction. It predicts the class substantially more often than it occurs (Prediction Bias score: {prediction_bias:.4f})."
                elif 0.05 <= prediction_bias < 0.1:
                    statistical_statement_pb = f"The model shows moderate overprediction. It has a noticeable tendency toward positive predictions (Prediction Bias score: {prediction_bias:.4f})."
                elif 0.02 <= prediction_bias < 0.05:
                    statistical_statement_pb = f"The model shows mild overprediction. It has a slight tendency to classify cases as positive (Prediction Bias score: {prediction_bias:.4f})."
                elif -0.02 <= prediction_bias < 0.02:
                    statistical_statement_pb = f"The model shows approximately unbiased prediction frequency. Predicted positive frequency is close to observed prevalence (Prediction Bias score: {prediction_bias:.4f})."
                elif -0.05 <= prediction_bias < -0.02:
                    statistical_statement_pb = f"The model shows mild underprediction. It predicts the class somewhat less frequently than observed (Prediction Bias score: {prediction_bias:.4f})."
                elif -0.10 <= prediction_bias < -0.05:
                    statistical_statement_pb = f"The model shows moderate underprediction. It systematically produces fewer positive predictions (Prediction Bias score: {prediction_bias:.4f})."
                else:
                    statistical_statement_pb = f"The model shows strong underprediction. It substantially under-identifies the class (Prediction Bias score: {prediction_bias:.4f})."

                statistical_statement.append(statistical_statement_pb)

                # Relative error burden
                fp_fn_ratio = fp / fn if fn > 0 else np.inf if fp > 0 else 0

                if fp_fn_ratio >= 4:
                    statistical_statement_fpfn = f"The error pattern is very strongly FP-dominant. Model produces substantially more false positives than missed positives (FP/FN ratio: {fp_fn_ratio:.4f})."
                elif 2 <= fp_fn_ratio < 4:
                    statistical_statement_fpfn = f"The error pattern is strongly FP-dominant. False positives clearly dominate (FP/FN ratio: {fp_fn_ratio:.4f})."
                elif 1.5 <= fp_fn_ratio < 2:
                    statistical_statement_fpfn = f"The error pattern is moderately FP-dominant. More false positives than false negatives (FP/FN ratio: {fp_fn_ratio:.4f})."
                elif 0.67 <= fp_fn_ratio < 1.5:
                    statistical_statement_fpfn = f"The error pattern is relatively balanced. FP and FN occur at broadly similar frequencies (FP/FN ratio: {fp_fn_ratio:.4f})."
                elif 0.5 <= fp_fn_ratio < 0.67:
                    statistical_statement_fpfn = f"The error pattern is moderately FN-dominant.More missed positives than false positives (FP/FN ratio: {fp_fn_ratio:.4f})."
                elif 0.25 <= fp_fn_ratio < 0.5:
                    statistical_statement_fpfn = f"The error pattern is strongly FN-dominant. False negatives clearly dominate (FP/FN ratio: {fp_fn_ratio:.4f})."
                else:
                    statistical_statement_fpfn = f"The error pattern is very strongly FN-dominant. Model misses substantially more positives than it falsely identifies (FP/FN ratio: {fp_fn_ratio:.4f})."

                statistical_statement.append(statistical_statement_fpfn)

            else:

                # Multiclass classification
                classes = np.unique(y_test)

                actual_prevalence = np.array(
                    [np.mean(y_test == cls) for cls in classes]
                )

                predicted_prevalence = np.array(
                    [np.mean(y_pred == cls) for cls in classes]
                )

                prediction_bias = predicted_prevalence - actual_prevalence
                class_statements = []
                for cls, bias in zip(classes, prediction_bias):

                    # Prediction Bias

                    if bias >= 0.1:
                        statistical_statement_pb = f"The model shows strong overprediction. It predicts the class {cls} substantially more often than it occurs (Prediction Bias score: {bias:+.4f})."
                    elif 0.05 <= bias < 0.1:
                        statistical_statement_pb = f"The model shows moderate overprediction. It predicts class {cls} noticeably more often than it occurs (Prediction Bias score: {bias:+.4f})."
                    elif 0.02 <= bias < 0.05:
                        statistical_statement_pb = f"The model shows mild overprediction. It predicts class {cls} slightly more often than it occurs (Prediction Bias score: {bias:+.4f})."
                    elif -0.02 <= bias < 0.02:
                        statistical_statement_pb = f"The model shows approximately unbiased prediction frequency. Predicted frequency for class {cls} is close to observed prevalence (Prediction Bias score: {bias:+.4f})."
                    elif -0.05 <= bias < -0.02:
                        statistical_statement_pb = f"The model shows mild underprediction. It predicts class {cls} somewhat less frequently than observed (Prediction Bias score: {bias:+.4f})."
                    elif -0.10 <= bias < -0.05:
                        statistical_statement_pb = f"The model shows moderate underprediction. It predicts class {cls} noticeably less frequently than observed (Prediction Bias score: {bias:+.4f})."
                    else:
                        statistical_statement_pb = f"The model shows strong underprediction. It substantially under-identifies the class {cls} (Prediction Bias score: {bias:+.4f})."

                    # FP/FN ratio

                    # One-vs-rest confusion counts for this class
                    tp = np.sum((y_test == cls) & (y_pred == cls))
                    fn = np.sum((y_test == cls) & (y_pred != cls))
                    fp = np.sum((y_test != cls) & (y_pred == cls))

                    # Relative error burden
                    fp_fn_ratio = fp / fn if fn > 0 else np.inf if fp > 0 else 0

                    if fp_fn_ratio >= 4:
                        statistical_statement_fpfn = (
                            f"The error pattern is very strongly FP-dominant. "
                            f"Class {cls} is predicted substantially more often incorrectly "
                            f"than it is missed (FP/FN ratio: {fp_fn_ratio:.4f})."
                        )
                    elif 2 <= fp_fn_ratio < 4:
                        statistical_statement_fpfn = (
                            f"The error pattern is strongly FP-dominant. "
                            f"Incorrect predictions of class {cls} clearly dominate missed cases "
                            f"(FP/FN ratio: {fp_fn_ratio:.4f})."
                        )
                    elif 1.5 <= fp_fn_ratio < 2:
                        statistical_statement_fpfn = (
                            f"The error pattern is moderately FP-dominant. "
                            f"Class {cls} has more false positives than false negatives "
                            f"(FP/FN ratio: {fp_fn_ratio:.4f})."
                        )
                    elif 0.67 <= fp_fn_ratio < 1.5:
                        statistical_statement_fpfn = (
                            f"The error pattern is relatively balanced for class {cls}. "
                            f"FP and FN occur at broadly similar frequencies "
                            f"(FP/FN ratio: {fp_fn_ratio:.4f})."
                        )
                    elif 0.5 <= fp_fn_ratio < 0.67:
                        statistical_statement_fpfn = (
                            f"The error pattern is moderately FN-dominant. "
                            f"Class {cls} has more false negatives than false positives "
                            f"(FP/FN ratio: {fp_fn_ratio:.4f})."
                        )
                    elif 0.25 <= fp_fn_ratio < 0.5:
                        statistical_statement_fpfn = (
                            f"The error pattern is strongly FN-dominant. "
                            f"False negatives for class {cls} clearly dominate "
                            f"(FP/FN ratio: {fp_fn_ratio:.4f})."
                        )
                    else:
                        statistical_statement_fpfn = (
                            f"The error pattern is very strongly FN-dominant. "
                            f"Class {cls} is missed substantially more often than it is falsely predicted "
                            f"(FP/FN ratio: {fp_fn_ratio:.4f})."
                        )

                    # Combine both statements for this class
                    class_statements.append(
                        f"Class {cls}: {statistical_statement_pb} "
                        f"{statistical_statement_fpfn}"
                    )
                statistical_statement.extend(class_statements)

            # Matthews Correlation Coefficient
            mcc = matthews_corrcoef(y_test, y_pred)

            if 0.9 <= mcc <= 1:
                statistical_statement_mcc = f"The model demonstrates a very strong correspondence between predicted and observed class status (Matthews Correlation Coefficient: {mcc:.4f})."
            elif 0.7 <= mcc < 0.9:
                statistical_statement_mcc = f"The model demonstrates a strong correspondence between predicted and observed class status (Matthews Correlation Coefficient: {mcc:.4f})."
            elif 0.5 <= mcc < 0.7:
                statistical_statement_mcc = f"The model demonstrates a moderate correspondence between predicted and observed class status (Matthews Correlation Coefficient: {mcc:.4f})."
            elif 0.3 <= mcc < 0.5:
                statistical_statement_mcc = f"The model demonstrates a limited correspondence between predicted and observed class status (Matthews Correlation Coefficient: {mcc:.4f})."
            elif 0.1 <= mcc < 0.3:
                statistical_statement_mcc = f"The model demonstrates a weak correspondence between predicted and observed class status (Matthews Correlation Coefficient: {mcc:.4f})."
            elif -0.1 <= mcc < 0.1:
                statistical_statement_mcc = f"The model demonstrates little meaningful correspondence between predicted and observed class status (Matthews Correlation Coefficient: {mcc:.4f})."
            elif mcc < -0.1:
                statistical_statement_mcc = f"The model demonstrates an inverse correspondence between prediction and observed class status. Predictions tend to disagree with the actual class (Matthews Correlation Coefficient: {mcc:.4f})."

            statistical_statement.append(statistical_statement_mcc)
            statistical_statement = "\n".join(statistical_statement)

    print(evaluation_statement)
    print(diagnostic_statement)
    print(statistical_statement)

    return evaluation_statement, diagnostic_statement, statistical_statement


if PROJECT_TYPE == "Regression":

    evaluation_statement, diagnostic_statement, statistical_statement = (
        result_analysis_summary(
            project_type=project_type, r2=r2, mae=mae, rmse=rmse, residuals=residuals
        )
    )

elif PROJECT_TYPE == "Classification":

    evaluation_statement, diagnostic_statement, statistical_statement = (
        result_analysis_summary(
            project_type=project_type,
            f1=f1,
            balanced_accuracy=balanced_accuracy,
            precision=precision,
            recall=recall,
            roc_auc=roc_auc,
            pr_auc=pr_auc,
            y_test=y_test,
            y_pred=y_pred,
            cm=cm,
        )
    )


experiment_summary = "\n".join(
    [
        f"# Experiment Summary: `{RUN_ID}`",
        "",
        "## Objective",
        "",
        "Provide a guided machine-learning workflow for selecting, optimizing, "
        "and evaluating a regression or classification model for a user-chosen "
        "dataset and target variable.",
        "",
        "## Experiment Results",
        "",
        "### Evaluation Metrics",
        "",
        evaluation_statement,
        "",
        "### Diagnostics",
        "",
        diagnostic_statement,
        "",
        "### Statistical Analysis",
        "",
        statistical_statement,
        "",
        "## Reproducibility",
        "",
        "| Artifact | Location |",
        "|---|---|",
        "| Configuration | `config/run_config.json` |",
        "| Exact split & random state | `manifests/split_manifest.csv` |",
        f"| Final model configuration | `checkpoints/model_{selected_configuration}_profile.joblib` |",
        "",
    ]
)


saved_paths_summary = REPORT_DIR / "experiment_summary.md"
saved_paths_summary.write_text(experiment_summary, encoding="utf-8")
print("\nExperiment summary saved successfully. Generated artifacts:")
print(f"  • {saved_paths_summary.name}")